**Note**: This notebook should be run from the `high-order-anesthesia` folder to ensure the correct imports and file paths are used.

In [1]:
from pathlib import Path
import os
def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(
        f"Could not find '{target_name}' in current path or parents. "
        f"Please run the notebook from inside the project."
    )
ROOT = ensure_project_root("high-order-anesthesia")
print(f"Now in: {ROOT.name}")


Now in: high-order-anesthesia


In [2]:
import torch
import math
import pickle
import gzip
from typing import List, Tuple

#### Custom libraries

In [ ]:
from src.hoi_anesthesia.io import load_covariance_dict, save_results
from src.hoi_anesthesia.utils import O_PR_AUC, generate_X, analyze_order,attach_delta_O_to_tail, build_X_for_dataset, build_region_maps_for_tail

In [4]:
results_path = "results"
data_path = "results"
all_covs = load_covariance_dict(f"{data_path}/covariance_matrices_gc.h5")
N = 82
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Np_max = 500_000
M = 50_000_000
R = 82
batch_size = 1000
p = 0.05  # 5% of the total number of nplets
eval_func = O_PR_AUC

MA: order 4 (synergy brain), order 7 (redundancy brain)

In [5]:
conscious_states = {"MA": ["MA_awake"]}
nonresponsive_states = {
    "MA": ["deep_propofol", "ketamine", "moderate_propofol", "ts_selv2", "ts_selv4"],
}
X_tensor, n_c, n_nr = generate_X(conscious_states, nonresponsive_states, all_covs)

torch.cuda.empty_cache()
for k in [3]:
    results = {"MA": {}}
    tail_size = int(min(math.comb(R, k) * p, Np_max))
    print(
        f"MA, order {k} | tail size: {tail_size:.0f} (of {math.comb(R, k):.0f} combinations)"
    )
    top_c, top_nr = analyze_order(
        X_tensor=X_tensor,
        n_c=n_c,
        k=k,
        M=M,
        Np=tail_size,
        batch_size=batch_size,
        R=R,
        device=device,
        eval_fc=eval_func,
    )
    # top_c: high O -> C (best PR with C positive)
    # top_nr: high O -> NR (best PR with NR positive)
    results["MA"][k] = {
        "C_positive": top_c,
        "NR_positive": top_nr,
    }
    save_results(results, f"{results_path}/nplet_tails_PRAUC_MA_{k}.pkl.gz")
torch.cuda.empty_cache()

MA, order 3 | tail size: 4428 (of 88560 combinations)
Generating 50000000 unique nplets...


Evaluating n-plets (k=3): 100%|██████████| 89/89 [01:23<00:00,  1.07it/s]


DBS: order 3 (synergy brain), order 9 (redundancy brain)

In [6]:
torch.cuda.empty_cache()

conscious_states = {"DBS": ["DBS_awake", "ts_on_5V"]}
nonresponsive_states = {
    "DBS": ["ts_off", "ts_on_3V_control", "ts_on_5V_control"],
}
X_tensor, n_c, n_nr = generate_X(conscious_states, nonresponsive_states, all_covs)

for k in [3]:
    results = {"DBS": {}}
    tail_size = int(min(math.comb(R, k) * p, Np_max))
    print(
        f"DBS, order {k} | tail size: {tail_size:.0f} (of {math.comb(R, k):.0f} combinations)"
    )
    top_c, top_nr = analyze_order(
        X_tensor=X_tensor,
        n_c=n_c,
        k=k,
        M=M,
        Np=tail_size,
        batch_size=batch_size,
        R=R,
        device=device,
        eval_fc=eval_func,
    )
    results["DBS"][k] = {
        "C_positive": top_c,
        "NR_positive": top_nr,
    }
    save_results(results, f"{results_path}/nplet_tails_PRAUC_DBS_{k}.pkl.gz")
torch.cuda.empty_cache()


DBS, order 3 | tail size: 4428 (of 88560 combinations)
Generating 50000000 unique nplets...


Evaluating n-plets (k=3): 100%|██████████| 89/89 [01:26<00:00,  1.03it/s]


In [7]:


# ---------------------------------------------------------------------
# Step 2 main: load all tails, merge, compute ΔO, save merged dict
# ---------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Load step-1 tails and merge into a single dict
merged_results = {"MA": {}, "DBS": {}}

# MA: orders 4 (redundant NRpos) and 7 (redundant Cpos)
for k in [3]:
    path = f"{results_path}/nplet_tails_PRAUC_MA_{k}.pkl.gz"
    print(f"Loading MA order {k} tails from: {path}")
    with gzip.open(path, "rb") as f:
        res = pickle.load(f)  # {"MA": {k: {"C_positive": [...], "NR_positive": [...]}}}
    merged_results["MA"][k] = res["MA"][k]

# DBS: orders 3 (synergy NRpos) and 9 (redundant Cpos)
for k in [3]:
    path = f"{results_path}/nplet_tails_PRAUC_DBS_{k}.pkl.gz"
    print(f"Loading DBS order {k} tails from: {path}")
    with gzip.open(path, "rb") as f:
        res = pickle.load(
            f
        )  # {"DBS": {k: {"C_positive": [...], "NR_positive": [...]}}}
    merged_results["DBS"][k] = res["DBS"][k]

# 2) Rebuild X_tensor and n_c for each dataset (same as step 1)


# 3) For each dataset/order/tail, attach ΔO

for dataset in ["MA", "DBS"]:
    print(f"\n=== Attaching ΔO for dataset: {dataset} ===")
    X_tensor, n_c = build_X_for_dataset(dataset,all_covs)
    X_tensor = X_tensor.to(device=device, dtype=torch.float64)

    for k, tails in merged_results[dataset].items():
        print(f"  Order k={k}")

        for key in ["C_positive", "NR_positive"]:
            tail = tails.get(key, [])
            if not tail:
                merged_results[dataset][k][key] = []
                continue

            print(f"    Tail: {key}, n={len(tail)}")
            tail_with_delta = attach_delta_O_to_tail(
                X_tensor=X_tensor,
                n_c=n_c,
                tail=tail,
                k=k,
                device=device,
                batch_size=2048,
            )
            # Now each element: (PR_AUC, ΔO, nplet_tuple)
            merged_results[dataset][k][key] = tail_with_delta

merged_results[dataset][k][key][0]

# 4) Save merged dict with ΔO attached for all datasets/orders/tails
out_path = f"{results_path}/B_2_nplet_tails_PRAUC_with_deltaO_ALL.pkl.gz"
print(f"\nSaving merged results with ΔO to: {out_path}")
save_results(merged_results, out_path)


Loading MA order 3 tails from: results/nplet_tails_PRAUC_MA_3.pkl.gz
Loading DBS order 3 tails from: results/nplet_tails_PRAUC_DBS_3.pkl.gz

=== Attaching ΔO for dataset: MA ===
  Order k=3
    Tail: C_positive, n=4428


Computing ΔO for tail (k=3, B=4428): 100%|██████████| 3/3 [00:00<00:00,  7.35it/s]


    Tail: NR_positive, n=4428


Computing ΔO for tail (k=3, B=4428): 100%|██████████| 3/3 [00:00<00:00, 45.10it/s]



=== Attaching ΔO for dataset: DBS ===
  Order k=3
    Tail: C_positive, n=4428


Computing ΔO for tail (k=3, B=4428): 100%|██████████| 3/3 [00:00<00:00, 32.78it/s]


    Tail: NR_positive, n=4428


Computing ΔO for tail (k=3, B=4428): 100%|██████████| 3/3 [00:00<00:00, 36.35it/s]



Saving merged results with ΔO to: results/B_2_nplet_tails_PRAUC_with_deltaO_ALL.pkl.gz


In [ ]:
import gzip
import pickle
import numpy as np



merged_path = f"{results_path}/B_2_nplet_tails_PRAUC_with_deltaO_ALL.pkl.gz"
print(f"Loading merged tails (with ΔO) from: {merged_path}")
with gzip.open(merged_path, "rb") as f:
    merged_results = pickle.load(f)
    # structure:
    # merged_results[dataset][k]["C_positive"] = [(pr_auc, delta_O, nplet), ...]
    # merged_results[dataset][k]["NR_positive"] = [(pr_auc, delta_O, nplet), ...]

# ---------------------------------------------------------------------
# Build region maps for all dataset / order / tail
# ---------------------------------------------------------------------

region_maps = {
    "MA": {},
    "DBS": {},
}

for dataset in ["MA", "DBS"]:
    print(f"\n=== Building region maps for dataset: {dataset} ===")

    for k, tails in merged_results[dataset].items():
        print(f"  Order k={k}")
        region_maps[dataset][k] = {}

        for key in ["C_positive", "NR_positive"]:
            tail_with_delta = tails.get(key, [])
            # tail_with_delta[0]
            print(f"    Tail: {key}, n_raw={len(tail_with_delta)}")

            (
                region_counts,
                region_counts_prop,
                region_counts_z,
                region_counts_percent,
            ) = build_region_maps_for_tail(
                tail_with_delta=tail_with_delta,
                mode=key,  # "C_positive" or "NR_positive"
                R=R,
                delta_eps=0.0,  # you can set >0 to be stricter on ΔO
            )

            region_maps[dataset][k][key] = {
                "region_counts": region_counts,
                "region_counts_prop": region_counts_prop,
                "region_counts_z": region_counts_z,
                "region_counts_percent": region_counts_percent,
            }


# ---------------------------------------------------------------------
# Save region_maps to disk for plotting
# ---------------------------------------------------------------------

out_maps_path = f"{results_path}/R1_C_region_maps_PRAUC_deltaO.pkl.gz"
print(f"\nSaving region maps to: {out_maps_path}")
with gzip.open(out_maps_path, "wb") as f:
    pickle.dump(region_maps, f, protocol=pickle.HIGHEST_PROTOCOL)

Loading merged tails (with ΔO) from: results/B_2_nplet_tails_PRAUC_with_deltaO_ALL.pkl.gz

=== Building region maps for dataset: MA ===
  Order k=3
    Tail: C_positive, n_raw=4428
      After ΔO filtering (C_positive): 4428 n-plets
    Tail: NR_positive, n_raw=4428
      After ΔO filtering (NR_positive): 4376 n-plets

=== Building region maps for dataset: DBS ===
  Order k=3
    Tail: C_positive, n_raw=4428
      After ΔO filtering (C_positive): 4428 n-plets
    Tail: NR_positive, n_raw=4428
      After ΔO filtering (NR_positive): 570 n-plets

Saving region maps to: results/R1_C_region_maps_PRAUC_deltaO.pkl.gz
